# 10: Profile & Branding System Testing

This notebook tests the complete Profile/Branding workflow:

1. **User Profile** - Create and persist user configuration
2. **Client Profile** - Create Hillcrest as a client with branding
3. **1Password Integration** - Get credentials from 1Password
4. **Branded Report** - Generate a PDF with client branding

## Prerequisites

- 1Password CLI (`op`) installed and authenticated
- Run on macOS for Keychain support

In [ ]:
import os
import sys
from pathlib import Path
from datetime import datetime

# Ensure siege_utilities is importable
sys.path.insert(0, str(Path.cwd().parent))

print("=== System Check ===")
print(f"Python: {sys.version}")
print(f"Working Directory: {Path.cwd()}")
print(f"Platform: {sys.platform}")

## 1. Check Credential Backends

Check which credential storage backends are available.

In [ ]:
from siege_utilities.config import CredentialManager, credential_status

# Check backend status
print("=== Credential Backend Status ===")
status = credential_status()

for backend, info in status.items():
    symbol = "✅" if info['available'] else "❌"
    print(f"{symbol} {backend}: {info['status']}")
    print(f"   {info['description']}")

In [ ]:
# Initialize credential manager
cred_manager = CredentialManager()

print("\n=== Credential Manager Initialized ===")
print(f"Backend priority: {cred_manager.backend_priority}")
print(f"Available backends: {[k for k,v in cred_manager.available_backends.items() if v]}")
print(f"Default vault: {cred_manager.default_vault}")

## 2. Create User Profile

Create or load user profile with default preferences.

In [ ]:
from siege_utilities.config.models import UserProfile
from siege_utilities.config import save_user_profile, load_user_profile, list_client_profiles

# Check for existing profiles
profiles_dir = Path.home() / ".siege_utilities" / "profiles"
users_dir = profiles_dir / "users"
clients_dir = profiles_dir / "clients"

print("=== Profile Directories ===")
print(f"Users: {users_dir} (exists: {users_dir.exists()})")
print(f"Clients: {clients_dir} (exists: {clients_dir.exists()})")

if users_dir.exists():
    existing_users = list(users_dir.glob("*.yaml"))
    print(f"\nExisting user profiles: {[f.stem for f in existing_users]}")

if clients_dir.exists():
    existing_clients = list_client_profiles()
    print(f"Existing client profiles: {existing_clients}")

In [ ]:
# Create user profile for Dheeraj
user_profile = UserProfile(
    username="dheeraj",
    email="dheeraj@siegeanalytics.com",
    full_name="Dheeraj Chand",
    github_login="dheerajchand",
    organization="Siege Analytics",
    preferred_download_directory=str(Path.home() / "Downloads" / "siege_utilities"),
    default_output_format="pdf",
    preferred_map_style="carto-positron",
    default_color_scheme="viridis",
    default_dpi=300,
    default_figure_size=[10, 8],
    enable_logging=True,
    log_level="INFO"
)

print("=== User Profile Created ===")
print(f"Username: {user_profile.username}")
print(f"Organization: {user_profile.organization}")
print(f"Default output: {user_profile.default_output_format}")

In [ ]:
# Save user profile
success = save_user_profile(user_profile, "dheeraj")
print(f"User profile saved: {success}")

# Verify by loading
loaded_profile = load_user_profile("dheeraj")
if loaded_profile:
    print(f"Verified: Loaded profile for {loaded_profile.username}")
else:
    print("Warning: Could not load saved profile")

## 3. Create Hillcrest Client Profile

Create a complete client profile with branding configuration.

In [ ]:
from siege_utilities.config.models import (
    ClientProfile, ContactInfo, BrandingConfig, ReportPreferences
)
from siege_utilities.config import save_client_profile, load_client_profile

# Create Hillcrest branding config
# (Update these colors to match Hillcrest's actual branding)
hillcrest_branding = BrandingConfig(
    # Brand colors - UPDATE THESE with Hillcrest's actual colors
    primary_color="#1a4d2e",      # Forest green (placeholder)
    secondary_color="#4a7c59",    # Sage green (placeholder)
    accent_color="#f4a261",       # Warm accent (placeholder)
    text_color="#2d3436",         # Dark gray
    background_color="#ffffff",   # White
    
    # Fonts
    primary_font="Helvetica",
    secondary_font="Georgia",
    
    # Logo (update with actual path when available)
    logo_path=None,  # Will be set when logo is available
    
    # Layout
    header_height=50,
    footer_height=25,
    margin_top=25,
    margin_bottom=25,
    margin_left=20,
    margin_right=20,
    
    # Typography
    title_font_size=28,
    subtitle_font_size=20,
    body_font_size=12,
    caption_font_size=10,
    
    # Charts
    chart_color_palette="viridis",
    chart_background_color="#ffffff",
    chart_grid_color="#e0e0e0"
)

print("=== Hillcrest Branding Config ===")
print(f"Primary color: {hillcrest_branding.primary_color}")
print(f"Secondary color: {hillcrest_branding.secondary_color}")
print(f"Fonts: {hillcrest_branding.primary_font} / {hillcrest_branding.secondary_font}")
print(f"\nColor scheme: {hillcrest_branding.get_color_scheme()}")

In [ ]:
# Create Hillcrest report preferences
hillcrest_reports = ReportPreferences(
    default_format="pdf",
    include_executive_summary=True,
    chart_style="professional",
    page_size="Letter",
    orientation="landscape",
    include_table_of_contents=True,
    include_page_numbers=True,
    chart_quality="high",
    chart_dpi=300,
    include_chart_titles=True,
    include_chart_legends=True,
    sections=["executive_summary", "methodology", "findings", "recommendations"]
)

print("=== Hillcrest Report Preferences ===")
print(f"Format: {hillcrest_reports.default_format}")
print(f"Page: {hillcrest_reports.page_size} {hillcrest_reports.orientation}")
print(f"Sections: {hillcrest_reports.sections}")

In [ ]:
# Create Hillcrest contact info (update with actual contact)
hillcrest_contact = ContactInfo(
    email="contact@hillcrest.example.com",  # UPDATE with real email
    phone="(555) 123-4567",                 # UPDATE with real phone
    website="https://www.hillcrest.example.com",  # UPDATE with real website
)

print("=== Hillcrest Contact Info ===")
print(f"Email: {hillcrest_contact.email}")
print(f"Phone: {hillcrest_contact.phone}")

In [ ]:
# Create the complete Hillcrest client profile
hillcrest_profile = ClientProfile(
    client_id="hillcrest",
    client_name="Hillcrest",
    client_code="HILL",
    contact_info=hillcrest_contact,
    industry="Political Consulting",  # UPDATE if different
    project_count=0,
    status="active",
    branding_config=hillcrest_branding,
    report_preferences=hillcrest_reports,
    notes="Primary client for Siege Analytics"
)

print("=== Hillcrest Client Profile ===")
print(hillcrest_profile.get_summary())

In [ ]:
# Save Hillcrest profile
success = save_client_profile(hillcrest_profile)
print(f"Hillcrest profile saved: {success}")

# Verify by loading
loaded_hillcrest = load_client_profile("HILL")
if loaded_hillcrest:
    print(f"\nVerified: Loaded profile for {loaded_hillcrest.client_name}")
    print(f"Client code: {loaded_hillcrest.client_code}")
    print(f"Primary color: {loaded_hillcrest.branding_config.primary_color}")
else:
    print("Warning: Could not load saved profile")

## 4. Test 1Password Integration

Retrieve credentials from 1Password (if available).

In [ ]:
from siege_utilities.config import get_google_service_account_from_1password

# Check if 1Password is available
if cred_manager.available_backends.get('1password'):
    print("=== 1Password Available ===")
    
    # List stored credentials with siege-utilities tag
    stored_creds = cred_manager.list_stored_credentials()
    print(f"\nFound {len(stored_creds)} stored credentials:")
    for cred in stored_creds:
        print(f"  - {cred.get('service', 'unknown')} ({cred.get('backend', 'unknown')})")
else:
    print("1Password not available - skipping 1Password tests")

In [ ]:
# Try to get Google Analytics service account from 1Password
# The item title should match what's stored in 1Password

ga_service_account = None

if cred_manager.available_backends.get('1password'):
    # Try common item titles
    item_titles = [
        "Google Analytics Service Account - Multi-Client Reporter",
        "Google Analytics Service Account",
        "GA4 Service Account",
        "Hillcrest GA Service Account"
    ]
    
    for title in item_titles:
        try:
            print(f"Trying: {title}...")
            ga_service_account = get_google_service_account_from_1password(title)
            if ga_service_account:
                print(f"✅ Found service account!")
                print(f"   Project: {ga_service_account.get('project_id')}")
                print(f"   Email: {ga_service_account.get('client_email')}")
                break
        except Exception as e:
            print(f"   Not found or error: {str(e)[:50]}")
            continue
    
    if not ga_service_account:
        print("\n⚠️ No GA service account found in 1Password")
        print("To store one, use:")
        print("  from siege_utilities.config import store_ga_service_account_from_file")
        print("  store_ga_service_account_from_file('/path/to/service-account.json')")
else:
    print("Skipping 1Password credential retrieval (not available)")

## 5. Generate Branded PDF Report

Create a sample report using Hillcrest branding.

In [ ]:
import pandas as pd
import numpy as np

# Create sample data for the report
np.random.seed(42)

# Sample polling data
poll_data = pd.DataFrame({
    'Candidate': ['Smith', 'Jones', 'Williams', 'Undecided'],
    'Support (%)': [42, 38, 12, 8],
    'Previous (%)': [40, 39, 13, 8],
    'Change': ['+2', '-1', '-1', '0']
})

# Sample demographic breakdown
demo_data = pd.DataFrame({
    'Age Group': ['18-29', '30-44', '45-59', '60+'],
    'Smith': [35, 40, 45, 48],
    'Jones': [45, 42, 35, 32],
    'Williams': [15, 12, 12, 10]
})

print("=== Sample Polling Data ===")
print(poll_data.to_string(index=False))
print("\n=== Demographic Breakdown ===")
print(demo_data.to_string(index=False))

In [ ]:
from siege_utilities.reporting.report_generator import ReportGenerator

# Create output directory
output_dir = Path.home() / "Downloads" / "siege_reports"
output_dir.mkdir(parents=True, exist_ok=True)

# Initialize ReportGenerator with Hillcrest branding
rg = ReportGenerator(
    client_name="Hillcrest",
    output_dir=output_dir
)

print(f"=== ReportGenerator Initialized ===")
print(f"Client: {rg.client_name}")
print(f"Output directory: {rg.output_dir}")

In [ ]:
# Build report content
report_content = {
    'sections': [],
    'metadata': {
        'title': 'Sample Polling Report',
        'subtitle': 'Q1 2026 Survey Results',
        'author': 'Siege Analytics',
        'date': datetime.now().strftime('%B %d, %Y'),
        'client': 'Hillcrest'
    }
}

# Add executive summary
executive_summary = """
This polling report presents findings from a survey of 1,200 likely voters 
conducted January 10-15, 2026. Key findings include:

• Smith leads with 42% support, up 2 points from previous poll
• Jones at 38%, down 1 point
• Williams at 12%, essentially unchanged
• 8% remain undecided with 3 weeks until election

Margin of error: ±2.8 percentage points (95% confidence)
"""

report_content = rg.add_text_section(
    report_content, 
    'Executive Summary', 
    executive_summary
)

print("Added executive summary")

In [ ]:
# Add polling data table
report_content = rg.add_table_section(
    report_content,
    'Current Poll Results',
    poll_data
)

print("Added polling data table")

In [ ]:
# Add demographic breakdown table
report_content = rg.add_table_section(
    report_content,
    'Support by Age Group',
    demo_data
)

print("Added demographic breakdown table")

In [ ]:
# Add methodology section
methodology = """
Survey Methodology:

• Sample Size: 1,200 likely voters
• Mode: Mixed (60% cell, 40% landline)
• Field Dates: January 10-15, 2026
• Weighting: Age, gender, education, party registration
• Margin of Error: ±2.8 percentage points

The survey was conducted by Siege Analytics using live interviewers. 
Respondents were selected using stratified random sampling from 
voter file records.
"""

report_content = rg.add_text_section(
    report_content,
    'Methodology',
    methodology
)

print("Added methodology section")

In [ ]:
# Generate the PDF
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_path = output_dir / f"hillcrest_poll_report_{timestamp}.pdf"

print(f"Generating PDF: {output_path}")

try:
    rg.generate_pdf_report(report_content, str(output_path))
    
    if output_path.exists():
        size_kb = output_path.stat().st_size / 1024
        print(f"\n✅ PDF Generated Successfully!")
        print(f"   Path: {output_path}")
        print(f"   Size: {size_kb:.1f} KB")
        print(f"   Sections: {len(report_content['sections'])}")
    else:
        print("\n❌ PDF was not created")
except Exception as e:
    print(f"\n❌ Error generating PDF: {e}")
    import traceback
    traceback.print_exc()

## 6. Verify Profile Persistence

Verify that profiles are correctly saved and can be reloaded.

In [ ]:
# Reload and verify all profiles
print("=== Verification: Reload All Profiles ===")

# User profile
user = load_user_profile("dheeraj")
if user:
    print(f"\n✅ User Profile: {user.username}")
    print(f"   Organization: {user.organization}")
    print(f"   Default format: {user.default_output_format}")
else:
    print("\n❌ User profile not found")

# Client profile
client = load_client_profile("HILL")
if client:
    print(f"\n✅ Client Profile: {client.client_name}")
    print(f"   Code: {client.client_code}")
    print(f"   Industry: {client.industry}")
    print(f"   Primary color: {client.branding_config.primary_color}")
    print(f"   Primary font: {client.branding_config.primary_font}")
else:
    print("\n❌ Client profile not found")

In [ ]:
# Check saved profile files
print("=== Profile Files ===")

profiles_base = Path.home() / ".siege_utilities" / "profiles"

for profile_type in ["users", "clients"]:
    profile_dir = profiles_base / profile_type
    if profile_dir.exists():
        files = list(profile_dir.glob("*.yaml"))
        print(f"\n{profile_type.title()}:")
        for f in files:
            size = f.stat().st_size
            print(f"  - {f.name} ({size} bytes)")
    else:
        print(f"\n{profile_type.title()}: Directory not found")

## 7. Summary

Test results summary.

In [ ]:
# Summary of all tests
print("="*50)
print("PROFILE & BRANDING SYSTEM TEST RESULTS")
print("="*50)

results = {
    "Credential Backends": "1password" in [k for k,v in cred_manager.available_backends.items() if v],
    "User Profile Created": load_user_profile("dheeraj") is not None,
    "Client Profile Created": load_client_profile("HILL") is not None,
    "1Password Integration": cred_manager.available_backends.get('1password', False),
    "PDF Generation": output_path.exists() if 'output_path' in dir() else False,
}

for test, passed in results.items():
    symbol = "✅" if passed else "❌"
    print(f"{symbol} {test}")

passed = sum(results.values())
total = len(results)
print(f"\n{passed}/{total} tests passed")

if passed == total:
    print("\n🎉 All Profile/Branding tests passed!")
else:
    print("\n⚠️ Some tests need attention")

## Next Steps

If all tests pass:

1. **Update Hillcrest branding colors** - Edit the BrandingConfig with actual Hillcrest colors
2. **Add logo** - Set `logo_path` to actual Hillcrest logo file
3. **Store GA credentials** - If you have GA service account, store it:
   ```python
   from siege_utilities.config import store_ga_service_account_from_file
   store_ga_service_account_from_file('/path/to/service-account.json', 
                                       item_title='Hillcrest GA Service Account')
   ```
4. **Test with real data** - Use actual polling data from Hillcrest projects